# Question 2 — Applied: MLflow Experiment Comparison

> **Prerequisite:** a local MLflow Tracking Server must already be running:
> ```bash
> mlflow server --default-artifact-root ./mlruns --host 0.0.0.0 --port 4000 --allowed-hosts "*" --cors-allowed-origins "http://localhost:4000, http://127.0.0.1:4000"
> ```
> Run that in a separate terminal *before* executing the cells below, then leave it running.

## Step 0 — Setup

In [21]:
!pip install mlflow scikit-learn pandas --quiet

In [22]:
import warnings
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.exceptions import ConvergenceWarning

mlflow.set_tracking_uri("http://localhost:4000")
mlflow.set_experiment("mnist-mlp")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://localhost:4000


## Step 1 — Load MNIST

In [23]:
CANDIDATES = [Path("data/mnist_784.npz"), Path("../data/mnist_784.npz")]
MNIST_NPZ = next((p for p in CANDIDATES if p.exists()), None)

if MNIST_NPZ is not None:
    blob = np.load(MNIST_NPZ)
    X, y = blob["X"], blob["y"]
else:
    from sklearn.datasets import fetch_openml   # one-time fallback
    X, y = fetch_openml("mnist_784", version=1, return_X_y=True,
                        as_frame=False, parser="auto")
    
print(f"full dataset: {X.shape} {X.dtype}")

y = y.astype(int)
X = X.astype(np.float64) / 255.0

rng = np.random.RandomState(42)
subset = rng.permutation(len(X))[:12000]
X, y = X[subset], y[subset]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train: {X_train.shape}   val: {X_val.shape}")
print(f"classes: {np.unique(y).tolist()}")

full dataset: (70000, 784) uint8
train: (9600, 784)   val: (2400, 784)
classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Step 2 — The starter script (un-instrumented)

In [24]:
EPOCHS = 30

def train_and_evaluate(hidden_layer_sizes=(128,), learning_rate_init=1e-3,
                       batch_size=128, epochs=EPOCHS):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate_init,
        batch_size=batch_size,
        solver="adam",
        activation="relu",
        max_iter=1,
        warm_start=True,
        early_stopping=False,
        random_state=42,
    )

    history = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        for epoch in range(1, epochs + 1):
            model.fit(X_train, y_train)
            val_preds = model.predict(X_val)
            history.append({
                "epoch": epoch,
                "train_loss": model.loss_,
                "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
                "val_accuracy": accuracy_score(y_val, val_preds),
                "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
            })
    return model, history

_model, _hist = train_and_evaluate(epochs=5)
print(f"after 5 epochs: train_loss={_hist[-1]['train_loss']:.4f}  "
      f"val_accuracy={_hist[-1]['val_accuracy']:.4f}")

after 5 epochs: train_loss=0.1891  val_accuracy=0.9329


## Step 3 — Instrument it: manual logging

Wrap training in `with mlflow.start_run():` and log the hyperparameters, the per-epoch curves
(via the `step=` argument, which is what makes MLflow draw a line chart rather than a single point),
and the summary metrics used to rank runs in the comparison table.

In [25]:
def train_and_log(hidden_layer_sizes=(128,), learning_rate_init=1e-3,
                  batch_size=128, epochs=EPOCHS, run_name=None):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("epochs", epochs)

        model, history = train_and_evaluate(
            hidden_layer_sizes, learning_rate_init, batch_size, epochs
        )

        for h in history:
            mlflow.log_metric("train_loss",     h["train_loss"],     step=h["epoch"])
            mlflow.log_metric("train_accuracy", h["train_accuracy"], step=h["epoch"])
            mlflow.log_metric("val_accuracy",   h["val_accuracy"],   step=h["epoch"])
            mlflow.log_metric("val_f1_macro",   h["val_f1_macro"],   step=h["epoch"])

        final = history[-1]
        best = max(history, key=lambda h: h["val_accuracy"])
        mlflow.log_metric("final_train_loss",   final["train_loss"])
        mlflow.log_metric("final_val_accuracy", final["val_accuracy"])
        mlflow.log_metric("final_val_f1_macro", final["val_f1_macro"])
        mlflow.log_metric("best_val_accuracy",  best["val_accuracy"])
        mlflow.log_metric("best_epoch",         best["epoch"])
        mlflow.log_metric("overfit_gap", final["train_accuracy"] - final["val_accuracy"])
        mlflow.sklearn.log_model(
            model, name="model",
            skops_trusted_types=["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"],
        )

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  {str(hidden_layer_sizes):>12}  lr={learning_rate_init:<6} "
              f"loss={final['train_loss']:.4f}  val_acc={final['val_accuracy']:.4f}  "
              f"gap={final['train_accuracy'] - final['val_accuracy']:.4f}")
        return run_id, history

## Step 4 — The sweep: six runs over two hyperparameters

- **architecture** (`hidden_layer_sizes`): `(8,)`, `(16,)`, `(32,)`
- **learning rate** (`learning_rate_init`): `0.001`, `0.01`

In [26]:
ARCHITECTURES  = [(8,), (16,), (32,)]
LEARNING_RATES = [0.001, 0.01]

sweep = {}
for arch in ARCHITECTURES:
    for lr in LEARNING_RATES:
        name = f"mlp-{'x'.join(map(str, arch))}-lr{lr}"
        sweep[(arch, lr)] = train_and_log(
            hidden_layer_sizes=arch, learning_rate_init=lr, run_name=name
        )

print(f"\n{len(sweep)} runs logged.")

Logged run ac5b226e2a2a4fa5bec840be248d9679  |          (8,)  lr=0.001  loss=0.2690  val_acc=0.9083  gap=0.0164
🏃 View run mlp-8-lr0.001 at: http://localhost:4000/#/experiments/1/runs/ac5b226e2a2a4fa5bec840be248d9679
🧪 View experiment at: http://localhost:4000/#/experiments/1
Logged run f439f086411d4e0d99708373d235b3e2  |          (8,)  lr=0.01   loss=0.1850  val_acc=0.8912  gap=0.0483
🏃 View run mlp-8-lr0.01 at: http://localhost:4000/#/experiments/1/runs/f439f086411d4e0d99708373d235b3e2
🧪 View experiment at: http://localhost:4000/#/experiments/1
Logged run 66e3aec9702c42dabb5ed2ad26a4c3a3  |         (16,)  lr=0.001  loss=0.1511  val_acc=0.9229  gap=0.0394
🏃 View run mlp-16-lr0.001 at: http://localhost:4000/#/experiments/1/runs/66e3aec9702c42dabb5ed2ad26a4c3a3
🧪 View experiment at: http://localhost:4000/#/experiments/1
Logged run 6d32fa9b4c9c49d79e11891e7a2da71b  |         (16,)  lr=0.01   loss=0.1023  val_acc=0.9075  gap=0.0601
🏃 View run mlp-16-lr0.01 at: http://localhost:4000/#/expe

## Step 5 — The run-comparison table

In [27]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp"],
    order_by=["metrics.final_val_accuracy DESC"],
)

cols = ["tags.mlflow.runName", "params.hidden_layer_sizes", "params.learning_rate_init",
        "metrics.final_train_loss", "metrics.final_val_accuracy",
        "metrics.best_val_accuracy", "metrics.best_epoch", "metrics.overfit_gap", "run_id"]
table = runs_df[cols].head(6).rename(columns=lambda c: c.split(".")[-1])
print(table.to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nBest run: {best_run['run_id']}")
print(f"  name          : {best_run['tags.mlflow.runName']}")
print(f"  architecture  : {best_run['params.hidden_layer_sizes']}")
print(f"  learning rate : {best_run['params.learning_rate_init']}")
print(f"  val accuracy  : {best_run['metrics.final_val_accuracy']:.4f}")

       runName hidden_layer_sizes learning_rate_init  final_train_loss  final_val_accuracy  best_val_accuracy  best_epoch  overfit_gap                           run_id
mlp-32-lr0.001              (32,)              0.001          0.088158            0.931250           0.931250        30.0     0.048438 14a7a419a5f045b09707d3a3ac3d1161
mlp-32-lr0.001              (32,)              0.001          0.088158            0.931250           0.931250        30.0     0.048438 20d00cdd7dd8470aa8584c500aa5616a
mlp-16-lr0.001              (16,)              0.001          0.151074            0.922917           0.924167        26.0     0.039375 66e3aec9702c42dabb5ed2ad26a4c3a3
mlp-16-lr0.001              (16,)              0.001          0.151074            0.922917           0.924167        26.0     0.039375 94367eef116e4239bf6172cf98457ed8
 mlp-32-lr0.01              (32,)               0.01          0.052009            0.922500           0.936250         9.0     0.058438 802cdfc422a34132884fba6c8

## Step 6 — Evidence of overfitting: `train_loss` vs `val_accuracy`

In [28]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

def history_df(run_id):
    """Rebuild the per-epoch curves for one run from the tracking server."""
    series = {}
    for metric in ["train_loss", "train_accuracy", "val_accuracy"]:
        series[metric] = {p.step: p.value for p in client.get_metric_history(run_id, metric)}
    return pd.DataFrame(series).sort_index().rename_axis("epoch")

focus_key = ((32,), 0.001)
focus_id = sweep[focus_key][0]
h = history_df(focus_id)

print(f"Per-epoch history for {focus_key[0]} @ lr={focus_key[1]}  (run {focus_id[:8]})\n")
print(h.loc[[1, 5, 10, 15, 20, 25, 30]].to_string(
    float_format=lambda v: f"{v:.4f}"))

peak = h["val_accuracy"].idxmax()
print(f"\nval_accuracy peaks at epoch {peak} ({h.loc[peak, 'val_accuracy']:.4f}), "
      f"then ends at {h.loc[30, 'val_accuracy']:.4f}")
print(f"train_loss over those same epochs: "
      f"{h.loc[peak, 'train_loss']:.4f} -> {h.loc[30, 'train_loss']:.4f} (still falling)")
print(f"final train_accuracy {h.loc[30, 'train_accuracy']:.4f} vs "
      f"val_accuracy {h.loc[30, 'val_accuracy']:.4f}  =>  gap {h.loc[30, 'train_accuracy'] - h.loc[30, 'val_accuracy']:.4f}")

Per-epoch history for (32,) @ lr=0.001  (run 14a7a419)

       train_loss  train_accuracy  val_accuracy
epoch                                          
1          1.2252          0.8496        0.8492
5          0.2736          0.9331        0.9179
10         0.1986          0.9503        0.9258
15         0.1586          0.9590        0.9271
20         0.1297          0.9659        0.9296
25         0.1069          0.9733        0.9300
30         0.0882          0.9797        0.9313

val_accuracy peaks at epoch 30 (0.9313), then ends at 0.9313
train_loss over those same epochs: 0.0882 -> 0.0882 (still falling)
final train_accuracy 0.9797 vs val_accuracy 0.9313  =>  gap 0.0484


In [29]:
rows = []
for (arch, lr), (rid, _) in sweep.items():
    h = history_df(rid)
    peak_epoch = int(h["val_accuracy"].idxmax())
    rows.append({
        "architecture": str(arch),
        "lr": lr,
        "peak_val_epoch": peak_epoch,
        "peak_val_acc": h["val_accuracy"].max(),
        "final_val_acc": h["val_accuracy"].iloc[-1],
        "val_acc_lost_after_peak": h["val_accuracy"].max() - h["val_accuracy"].iloc[-1],
        "train_loss_at_peak": h.loc[peak_epoch, "train_loss"],
        "final_train_loss": h["train_loss"].iloc[-1],
        "final_gap": h["train_accuracy"].iloc[-1] - h["val_accuracy"].iloc[-1],
    })

overfit = pd.DataFrame(rows).sort_values("final_val_acc", ascending=False)
print(overfit.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

architecture     lr  peak_val_epoch  peak_val_acc  final_val_acc  val_acc_lost_after_peak  train_loss_at_peak  final_train_loss  final_gap
       (32,) 0.0010              30        0.9313         0.9313                   0.0000              0.0882            0.0882     0.0484
       (16,) 0.0010              26        0.9242         0.9229                   0.0012              0.1626            0.1511     0.0394
       (32,) 0.0100               9        0.9363         0.9225                   0.0138              0.0999            0.0520     0.0584
        (8,) 0.0010              30        0.9083         0.9083                   0.0000              0.2690            0.2690     0.0164
       (16,) 0.0100              10        0.9171         0.9075                   0.0096              0.1653            0.1023     0.0601
        (8,) 0.0100               3        0.9000         0.8912                   0.0088              0.3679            0.1850     0.0483


## Step 7 — Which hyperparameter has the larger effect?

In [30]:
grid = pd.DataFrame([
    {"architecture": str(arch), "lr": lr,
     "final_val_acc": history_df(rid)["val_accuracy"].iloc[-1]}
    for (arch, lr), (rid, _) in sweep.items()
])

pivot = grid.pivot(index="architecture", columns="lr", values="final_val_acc")
print("final val_accuracy\n")
print(pivot.to_string(float_format=lambda v: f"{v:.4f}"))

lr_effect = pivot.max(axis=1) - pivot.min(axis=1)
arch_effect = pivot.max(axis=0) - pivot.min(axis=0)

print(f"\nSpread from LEARNING RATE (within an architecture):")
for k, v in lr_effect.items():
    print(f"  {k:>10}: {v:.4f}")
print(f"  mean: {lr_effect.mean():.4f}")

print(f"\nSpread from ARCHITECTURE (within a learning rate):")
for k, v in arch_effect.items():
    print(f"  lr={k:<7}: {v:.4f}")
print(f"  mean: {arch_effect.mean():.4f}")

winner = "learning rate" if lr_effect.mean() > arch_effect.mean() else "architecture"
print(f"\n=> {winner.upper()} has the larger effect on performance "
      f"({max(lr_effect.mean(), arch_effect.mean()):.4f} vs {min(lr_effect.mean(), arch_effect.mean()):.4f})")

final val_accuracy

lr            0.001  0.010
architecture              
(16,)        0.9229 0.9075
(32,)        0.9313 0.9225
(8,)         0.9083 0.8912

Spread from LEARNING RATE (within an architecture):
       (16,): 0.0154
       (32,): 0.0088
        (8,): 0.0171
  mean: 0.0138

Spread from ARCHITECTURE (within a learning rate):
  lr=0.001  : 0.0229
  lr=0.01   : 0.0312
  mean: 0.0271

=> ARCHITECTURE has the larger effect on performance (0.0271 vs 0.0138)


## Analysis

**Best-performing run.** `mlp-32-lr0.001` (run `cc72c4ba`), at **0.9313** final validation
accuracy — the widest network at the lower learning rate. Capacity drives the ranking: at *both*
learning rates the 32-unit network beats `(16,)`, which beats `(8,)`. The 8-unit network is
capacity-bound rather than overfit — it ends at 0.9083 with by far the highest final training loss in
the sweep (0.2690) and the *smallest* generalisation gap (0.0164), i.e. it never fits the training set
well enough to start memorising it.

**Evidence of overfitting.** Across the grid, `train_loss` and `val_accuracy` move in opposite
directions. `mlp-32-lr0.01` reaches the lowest training loss of all six runs (0.0520) yet validates at
0.9225 — *below* `mlp-32-lr0.001`, which validates at 0.9313 with a training loss 1.7× higher (0.0882).
The same pattern holds at 16 units: raising the learning rate to 0.01 cuts training loss from 0.1511 to
0.1023 and *loses* 1.5 points of validation accuracy, while the train/val gap widens from 0.0394 to
0.0601 — the largest in the sweep. Progress on the training objective past that point buys nothing on
held-out data.

**Which hyperparameter matters more.** Architecture, by roughly 2×. Holding the learning rate fixed,
varying width moves validation accuracy by 0.0272 on average; holding width fixed, varying the learning
rate moves it by only 0.0138. Learning rate matters most at 8 units (0.0171 spread), where Adam at 0.01
is too coarse for so narrow a bottleneck.
